In [12]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
import rasterio
import albumentations as A
import timm
from tqdm import tqdm
import csv
from torch.cuda.amp import autocast, GradScaler

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [14]:
class CloudDataset(Dataset):
    def __init__(self, metadata, root_dir, transform=None):
        self.metadata = metadata
        self.root_dir = root_dir
        self.transform = transform
        self.feature_dir = os.path.join(root_dir, 'train_features')
        self.label_dir = os.path.join(root_dir, 'train_labels')

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        chip_id = self.metadata.iloc[idx]['chip_id']
        feature_path = os.path.join(self.feature_dir, chip_id)
        label_path = os.path.join(self.label_dir, f"{chip_id}.tif")

        bands = ['B02', 'B03', 'B04', 'B08']
        features = []

        for band in bands:
            with rasterio.open(os.path.join(feature_path, f'{band}.tif')) as src:
                features.append(src.read(1).astype(np.float32))

        image = np.stack(features, axis=-1)

        with rasterio.open(label_path) as src:
            mask = src.read(1).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        image = torch.from_numpy(image).permute(2, 0, 1).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()

        return image, mask, chip_id

In [15]:
mean_vals = (0.5, 0.5, 0.5, 0.5)
std_vals = (0.5, 0.5, 0.5, 0.5)

train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=mean_vals, std=std_vals, max_pixel_value=65535.0),
], additional_targets={'mask': 'mask'})

val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=mean_vals, std=std_vals, max_pixel_value=65535.0),
], additional_targets={'mask': 'mask'})


In [16]:
root_dir = 'data'
metadata = pd.read_csv('data/train_metadata.csv')

train_meta, temp_meta = train_test_split(metadata, test_size=0.3, random_state=42)
val_meta, test_meta = train_test_split(temp_meta, test_size=2/3, random_state=42)


In [17]:
train_dataset = CloudDataset(train_meta, root_dir, train_transform)
val_dataset   = CloudDataset(val_meta, root_dir, val_transform)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))


Train samples: 8223
Val samples: 1175


In [18]:
class XceptionBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model(
            'xception41', pretrained=True, features_only=True, in_chans=4
        )

    def forward(self, x):
        features = self.model(x)
        return features[1], features[3]
class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )

    def forward(self, x):
        return self.conv(x)
class ImprovedDeepLabV3Plus(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = XceptionBackbone()
        self.aspp = ASPP(1024, 256)
        self.final = nn.Conv2d(256, 1, 1)

    def forward(self, x):
        h, w = x.shape[-2:]
        low, high = self.backbone(x)
        x = self.aspp(high)
        x = F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False)
        return torch.sigmoid(self.final(x))


In [19]:
# Recreate model architecture
model = ImprovedDeepLabV3Plus()

# Load trained weights
model.load_state_dict(
    torch.load("final1_model2.pth", map_location=device)
)

model = model.to(device)
model.eval()


ImprovedDeepLabV3Plus(
  (backbone): XceptionBackbone(
    (model): FeatureHookNet(
      (stem_0): ConvNormAct(
        (conv): Conv2d(4, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNormAct2d(
          32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU(inplace=True)
        )
      )
      (stem_1): ConvNormAct(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNormAct2d(
          64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU(inplace=True)
        )
      )
      (blocks_0): XceptionModule(
        (shortcut): ConvNormAct(
          (conv): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
          (bn): BatchNormAct2d(
            128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()


In [20]:
def cloud_cover_ratio(mask):
    """
    mask: numpy array (H, W) with values 0 or 1
    """
    total_pixels = mask.size
    cloud_pixels = np.sum(mask)
    return (cloud_pixels / total_pixels) * 100
    
def solar_irradiance_from_cloud(mask, I_max=1000.0, alpha=0.8):
    """
    mask   : binary cloud mask (H, W), 1 = cloud, 0 = clear
    I_max  : maximum clear-sky solar irradiance (W/m²)
             (≈1000 W/m² at noon)
    alpha  : cloud attenuation factor (0.6–0.9 recommended)

    returns:
        cloud_ratio (%),
        estimated solar irradiance (W/m²)
    """
    cloud_ratio = cloud_cover_ratio(mask) / 100.0  # convert to 0–1
    irradiance = I_max * (1 - alpha * cloud_ratio)
    irradiance = max(irradiance, 0)  # safety clamp

    return cloud_ratio * 100, irradiance

In [21]:
def to_rgb(image_4band):
    """
    image_4band: (4, H, W)
    returns: (H, W, 3)
    """
    rgb = image_4band[[2, 1, 0], :, :]  # B04, B03, B02
    rgb = np.transpose(rgb, (1, 2, 0))

    # Normalize for display
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    return rgb


In [ ]:
import os
import csv
import torch

# ---------------------------
# Settings
# ---------------------------
I_MAX = 1000.0   # W/m²
ALPHA = 0.8

CSV_PATH = "Results/final1_model2/cloud_solar_metrics.csv"
os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)

# ---------------------------
# Sort dataset by chip_id
# ---------------------------
indexed_chips = [(val_dataset[i][2], i) for i in range(len(val_dataset))]
indexed_chips.sort(key=lambda x: x[0])   # alphabetical by chip_id

num_images = min(50, len(indexed_chips))
indices = [idx for _, idx in indexed_chips[:num_images]]

# ---------------------------
# Storage for CSV
# ---------------------------
results = []

# ---------------------------
# Inference + Metrics
# ---------------------------
model.eval()

with torch.no_grad():
    for idx in indices:
        # ---- Load sample ----
        image, mask, chip_id = val_dataset[idx]   # image: (4,H,W), mask: (1,H,W)

        image_t = image.unsqueeze(0).to(device)

        # ---- Model prediction ----
        output = model(image_t)
        pred = (output > 0.5).float()

        # ---- Move to CPU ----
        gt = mask[0].cpu().numpy()
        pr = pred[0, 0].cpu().numpy()

        # ---- Cloud cover ----
        gt_ratio = cloud_cover_ratio(gt)
        pr_ratio = cloud_cover_ratio(pr)

        # ---- Solar irradiance ----
        _, gt_irr = solar_irradiance_from_cloud(gt, I_max=I_MAX, alpha=ALPHA)
        _, pr_irr = solar_irradiance_from_cloud(pr, I_max=I_MAX, alpha=ALPHA)

        # ---- Save row ----
        results.append({
            "chip_id": chip_id,
            "gt_cloud_cover_percent": gt_ratio,
            "pred_cloud_cover_percent": pr_ratio,
            "gt_solar_irradiance_wm2": gt_irr,
            "pred_solar_irradiance_wm2": pr_irr
        })

# ---------------------------
# Write CSV
# ---------------------------
with open(CSV_PATH, mode="w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "chip_id",
            "gt_cloud_cover_percent",
            "pred_cloud_cover_percent",
            "gt_solar_irradiance_wm2",
            "pred_solar_irradiance_wm2",
        ]
    )
    writer.writeheader()
    writer.writerows(results)

print(f"CSV saved to: {CSV_PATH}")
